# Accelerating As-of Joins on GPU

As-of joins are a foundational building block in time-series analytics, powering many of the most data-intensive workflows in finance and beyond.
Whether that be for trade-quote alignment, order book reconstruction, signal and feature alignment, protfolio valuation/risk snapshots, or IoT and sensor data synchronization, the KDB-X GPU Acceleration utilizes `.gpu.aj` to parallelize binary searches across every symbol simultaneously, using thousands of GPU cores.

In this tutorial, we demonstrate performance differences of joining trade and quote data on GPU vs CPU.

The core function we expose is `.gpu.aj` which is called as follows:
```q
.gpu.aj[c;t1;t2]
```
where
- `t1` is a table
- `t2` is a table
- `c` is a symbol list of n column names, common to `t1` and `t2`, and of matching type
- column `c[n]` is of a sortable type (typically time)

It's important to note that:
- columns in the list `c` may be mapped to the gpu for improved performance
- the list of values `c` can be of length 2 at most
- if `c` is of length 2, the only attribute supported on c[0] is the grouped attribute ``g#`

Data can be transferred between CPU and GPU as follows:
```q
.gpu.to t
```

To map only specific columns to the gpu (`time` and `sym` here), use `.gpu.xto`:
```q
.gpu.xto[`time`sym] t
```

## Setup

This notebook is compatible with Linux, Mac, and Windows via WSL.

This notebook is designed to be run as a Python notebook, but the same functionality can be applied directly in a CLI with KDB-X (given it's running on a KDB-X CUDA application environment).
If running `q` from the terminal, just copy the `q` code and paste into your terminal.

### Data

In order to showcase the gpu capabilities in this tutorial, we utilize the [KX datagen module](https://code.kx.com/kdb-x/modules/datagen/overview.html) to generate synthetic trade and quote data.

To get started, we first want to import PyKX and define secondary thread count (mirroring the QARGS value the container was run with). We can then enable 'q first' mode which allows for running q code directly in notebook cells, similar to a q terminal.

In [1]:
## import pykx module
import pykx as kx

## define values for our q process
kx.Identity(kx.q('::'))

pykx.Identity(pykx.q('::'))

In [2]:
kx.util.jupyter_qfirst_enable()

KDB-X Python now running in 'jupyter_qfirst' mode. All cells by default will be run as q code.
Include '%%py' at the beginning of each cell to run as python code. 


Now that the Jupyter environment has been set up, let's load in order GPU module and leverage datagen to generate in-memory tables:

In [3]:
// GPU module initialization
// This can take any namespace, but we will use .gpu for clarity
.gpu:use`kx.gpu

// datagen module for capital market data
// argument to getInMemoryTables specifies the approximate number of trades per day
// increasing this value may throw a `wsfull error throughout the tutorial
([getInMemoryTables; buildPersistedDB]): use `kx.datagen.capmkts
(trade; quote; nbbo; master; exnames): getInMemoryTables 1000000

In [4]:
// Let's quickly inspect the trade and quote tables that have been created
count trade
3#trade
count quote
3#quote

1349688
sym  time                 price size stop cond ex
-------------------------------------------------
T    0D09:30:00.000003616 18    78   1           
TXN  0D09:30:00.000005843 18    44   0    4      
SOFI 0D09:30:00.000009678 214   75   0    7      
13500529
sym  time                 bid    ask    bsize asize mode ex
-----------------------------------------------------------
IBM  0D09:30:00.000000102 41.61  42.87  22    99    Z    S 
AAPL 0D09:30:00.000000191 83.52  84.68  70    54    8    D 
NVDA 0D09:30:00.000000253 131.41 132.11 48    87    K    W 


For as-of joins (`aj`):
- `.gpu.aj` API will automatically transfer to and from for CPU resident tables
- Better performance is acheived by leaving target columns resident on GPU using `.gpu.xto`
- Limited to 2 resident columns on GPU

Let's test this performance out:

In [5]:
"testing typical aj on a single column:"
\t aj[`time;trade;quote]
"aj on multiple columns:"
\t aj[`sym`time;trade;quote]

// can call against tables as is - this is a complete round trip
"\nround-trip aj on GPU:"
\t .gpu.aj[`time;trade;quote]
\t .gpu.aj[`sym`time;trade;quote]

testing typical aj on a single column:
233
aj on multiple columns:
360

round-trip aj on GPU:
73
105


The above `.gpu.aj` is a round-trip as it transfers to and from CPU - nothing is GPU-resident here. We can further optimize performance by sending join columns to GPU, as it saves on transfer costs:

In [6]:
Q:.gpu.xto[`time`sym] quote
T:.gpu.xto[`time`sym] trade

"aj for GPU-resident columns:"
\t .gpu.aj[`time;T;Q]
\t .gpu.aj[`sym`time;T;Q]

aj for GPU-resident columns:
40
42


It's important to understand the structure of the table here - we aren't paying the cost of transferring tables across PCIe. This structure can be examined using `.gpu.meta`

We move only the columns that matter for the operation, and the GPU does the work on those.

Also notice how the `sym` columns have the grouped attribute ``g#` applied to them - this is used to enable efficient as-of joins:

In [7]:
.gpu.meta T
.gpu.meta Q

c    | t f a r  
-----| ---------
sym  | s   g gpu
time | n   s gpu
price| f     cpu
size | j     cpu
stop | b     cpu
cond | c     cpu
ex   | s     cpu
c    | t f a r  
-----| ---------
sym  | s   g gpu
time | n   s gpu
bid  | f     cpu
ask  | f     cpu
bsize| j     cpu
asize| j     cpu
mode | c     cpu
ex   | s     cpu


We are super fast if everything is on GPU - this is a use-case for `.gpu.to`, and fully optimizes as-of joins on GPU:

In [8]:
tt:.gpu.to trade
qq:.gpu.to quote

"Structure when the entire table is moved to GPU:"
.gpu.meta tt
.gpu.meta qq

"\nNow we are completely optimized for GPU-enabled ajs:"
\t .gpu.aj[`sym`time;tt;qq].gpu.sync[]

Structure when the entire table is moved to GPU:
c    | t f a r  
-----| ---------
sym  | s   g gpu
time | n   s gpu
price| f     gpu
size | j     gpu
stop | b     gpu
cond | c     gpu
ex   | s     gpu
c    | t f a r  
-----| ---------
sym  | s   g gpu
time | n   s gpu
bid  | f     gpu
ask  | f     gpu
bsize| j     gpu
asize| j     gpu
mode | c     gpu
ex   | s     gpu

Now we are completely optimized for GPU-enabled ajs:
0
